# 🍎 FRESH ML: Fixed Fruit Classification Training for Kaggle

## Critical Issues Identified & Fixed:
- ❌ **Class Mapping Mismatch**: Model trained with ImageFolder alphabetical order vs custom dataset order
- ❌ **Config Key Errors**: Training script uses wrong YAML keys (`nc` vs `num_classes`, `names` vs `classes`)  
- ❌ **Green Mango Misclassification**: Due to incorrect class index mappings

## This Notebook:
✅ Fixes class mapping to match dataset.yaml configuration  
✅ Uses correct YAML configuration keys  
✅ Implements proper data augmentation for fruit classification  
✅ Includes comprehensive evaluation and monitoring  
✅ Designed for Kaggle GPU training environment  

**Dataset**: 16-class fruit classification with ripeness levels  
**Model**: ResNet50 with custom classification head  
**Classes**: mango (5), orange (4), guava (5), grapefruit (2)

## 1. Import Required Libraries

In [ ]:
# Essential imports for Kaggle environment
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Core ML libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models
import torchvision.transforms.functional as TF

# Data processing
import pandas as pd
import numpy as np
import yaml
import json
from pathlib import Path
import shutil
from collections import defaultdict, OrderedDict

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Metrics and evaluation
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    accuracy_score, precision_recall_fscore_support
)

# Utils
import time
import random
from datetime import datetime
import glob

# Set random seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

print("✅ All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory // 1e9:.1f} GB")

## 2. Load Dataset and Configuration with Fixed Mappings

In [ ]:
# Dataset configuration - FIXED VERSION
# This fixes the critical class mapping mismatch issue

DATASET_CONFIG = {
    'name': 'FRESH_ML_Unified_Fruit_Classification',
    'num_classes': 16,
    'classes': {
        0: 'mango_unripe',
        1: 'mango_early_ripe', 
        2: 'mango_partially_ripe',
        3: 'mango_ripe',
        4: 'mango_rotten',
        5: 'orange_unripe',
        6: 'orange_ripe',
        7: 'orange_rotten',
        8: 'orange_general',
        9: 'guava_unripe',
        10: 'guava_ripe',
        11: 'guava_overripe',
        12: 'guava_rotten',
        13: 'guava_general',
        14: 'grapefruit_pink',
        15: 'grapefruit_white'
    },
    'class_to_idx': {
        'mango_unripe': 0,
        'mango_early_ripe': 1,
        'mango_partially_ripe': 2,
        'mango_ripe': 3,
        'mango_rotten': 4,
        'orange_unripe': 5,
        'orange_ripe': 6,
        'orange_rotten': 7,
        'orange_general': 8,
        'guava_unripe': 9,
        'guava_ripe': 10,
        'guava_overripe': 11,
        'guava_rotten': 12,
        'guava_general': 13,
        'grapefruit_pink': 14,
        'grapefruit_white': 15
    }
}

# Key dataset paths - FIXED FOR YOUR KAGGLE STRUCTURE!
DATASET_PATH = "/kaggle/input/fresh-ml-fruit-classification/fruit_classification"  # ← Added /fruit_classification
TRAIN_PATH = f"{DATASET_PATH}/train"
VAL_PATH = f"{DATASET_PATH}/val" 
TEST_PATH = f"{DATASET_PATH}/test"

# Training configuration
CONFIG = {
    'batch_size': 32,
    'epochs': 50,
    'learning_rate': 0.001,
    'model_type': 'resnet50',
    'img_size': 224,
    'num_workers': 2,  # Kaggle limitation
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

print("✅ Dataset configuration loaded with FIXED class mappings")
print(f"Classes: {CONFIG['batch_size']} batch size, {CONFIG['epochs']} epochs")
print(f"Device: {CONFIG['device']}")
print(f"📁 Dataset path: {DATASET_PATH}")

# Display class mapping
print("\n📋 CORRECTED Class Mappings:")
for idx, class_name in DATASET_CONFIG['classes'].items():
    fruit_type = class_name.split('_')[0]
    ripeness = '_'.join(class_name.split('_')[1:]) if '_' in class_name else 'general'
    print(f"  {idx:2d}: {class_name:<20} ({fruit_type} - {ripeness})")

# Identify mango classes specifically
mango_classes = {k: v for k, v in DATASET_CONFIG['classes'].items() if 'mango' in v}
print(f"\n🥭 Mango Classes (5 total): {list(mango_classes.values())}")
print(f"   Unripe: {DATASET_CONFIG['class_to_idx']['mango_unripe']}")
print(f"   Ripe: {DATASET_CONFIG['class_to_idx']['mango_ripe']}")
print(f"   Early Ripe: {DATASET_CONFIG['class_to_idx']['mango_early_ripe']}")
print(f"   Partially Ripe: {DATASET_CONFIG['class_to_idx']['mango_partially_ripe']}")
print(f"   Rotten: {DATASET_CONFIG['class_to_idx']['mango_rotten']}")

In [ ]:
class CustomImageFolder(datasets.ImageFolder):
    """
    Custom ImageFolder that respects our specific class ordering
    instead of alphabetical ordering that caused the mapping issue
    """
    def __init__(self, root, transform=None, target_transform=None, class_to_idx=None):
        # First initialize with custom class_to_idx
        if class_to_idx is not None:
            # Store custom mapping
            self.custom_class_to_idx = class_to_idx
            
        super().__init__(root, transform, target_transform)
        
        # Override the class mappings with our custom ones
        if class_to_idx is not None:
            # Verify all classes exist
            folder_classes = set(os.listdir(root))
            config_classes = set(class_to_idx.keys())
            
            missing_folders = config_classes - folder_classes
            if missing_folders:
                print(f"⚠️  Missing folders: {missing_folders}")
            
            extra_folders = folder_classes - config_classes
            if extra_folders:
                print(f"⚠️  Extra folders (will be ignored): {extra_folders}")
                
            # Update class mappings
            self.class_to_idx = class_to_idx
            self.classes = [class_name for class_name, _ in sorted(class_to_idx.items(), key=lambda x: x[1])]
            
            # Rebuild samples with correct indices
            self.samples = []
            self.targets = []
            
            for class_name, class_index in class_to_idx.items():
                class_path = os.path.join(root, class_name)
                if os.path.isdir(class_path):
                    for fname in os.listdir(class_path):
                        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                            path = os.path.join(class_path, fname)
                            self.samples.append((path, class_index))
                            self.targets.append(class_index)
                            
            print(f"✅ CustomImageFolder: {len(self.samples)} samples loaded with correct class mapping")

# Data transforms with proper augmentation for fruit classification - FIXED!
def get_transforms():
    train_transforms = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(CONFIG['img_size'], scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.2),  # Fruits can be upside down
        transforms.RandomRotation(degrees=30),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ToTensor(),  # Convert to tensor FIRST
        transforms.Lambda(lambda x: x + torch.randn_like(x) * 0.01),  # Add noise AFTER tensor conversion
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_test_transforms = transforms.Compose([
        transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    return train_transforms, val_test_transforms

print("✅ Custom dataset class and transforms defined")

## 3. Load Existing Model for Evaluation

In [ ]:
def create_model(num_classes=16, model_type='resnet50', pretrained=True):
    """Create model with proper architecture"""
    if model_type == 'resnet50':
        model = models.resnet50(pretrained=pretrained)
        # Freeze early layers for transfer learning
        for param in list(model.parameters())[:-20]:  # Unfreeze more layers for better performance
            param.requires_grad = False
        # Replace final layer
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        
    elif model_type == 'efficientnet_b0':
        model = models.efficientnet_b0(pretrained=pretrained)
        for param in list(model.parameters())[:-20]:
            param.requires_grad = False
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        
    else:
        raise ValueError(f"Unsupported model type: {model_type}")
    
    return model

def load_existing_model(model_path=None):
    """
    Load existing model if available.
    Returns model and whether it needs retraining.
    """
    needs_retraining = True
    model = None
    model_info = {}
    
    # Try to load from Kaggle input or create new
    if model_path and os.path.exists(model_path):
        try:
            print(f"📂 Loading existing model from: {model_path}")
            
            # Load checkpoint
            checkpoint = torch.load(model_path, map_location='cpu')
            
            if isinstance(checkpoint, dict):
                if 'model_state_dict' in checkpoint:
                    state_dict = checkpoint['model_state_dict']
                    model_info = {k: v for k, v in checkpoint.items() if k != 'model_state_dict'}
                else:
                    state_dict = checkpoint
            else:
                state_dict = checkpoint
            
            # Create model architecture
            model = create_model(DATASET_CONFIG['num_classes'], CONFIG['model_type'])
            
            # Try to load state dict
            try:
                model.load_state_dict(state_dict, strict=True)
                print("✅ Model loaded successfully with strict matching")
                needs_retraining = False
            except Exception as e:
                print(f"⚠️  Strict loading failed: {e}")
                try:
                    model.load_state_dict(state_dict, strict=False)
                    print("⚠️  Model loaded with non-strict matching - may need retraining")
                    needs_retraining = True
                except Exception as e2:
                    print(f"❌ Failed to load model: {e2}")
                    model = None
                    needs_retraining = True
                    
        except Exception as e:
            print(f"❌ Error loading model: {e}")
            model = None
            needs_retraining = True
    else:
        print("🆕 No existing model found - will train from scratch")
        needs_retraining = True
    
    # Create new model if loading failed
    if model is None:
        print("🔧 Creating new model architecture...")
        model = create_model(DATASET_CONFIG['num_classes'], CONFIG['model_type'])
        
    return model, needs_retraining, model_info

# Try to load existing model
EXISTING_MODEL_PATH = "/kaggle/input/fresh-ml-models/classification_best.pth"  # Adjust path
existing_model, needs_retraining, model_info = load_existing_model(EXISTING_MODEL_PATH)

if existing_model:
    print(f"📊 Model architecture:")
    print(f"   Type: {CONFIG['model_type']}")
    print(f"   Parameters: {sum(p.numel() for p in existing_model.parameters()):,}")
    print(f"   Trainable: {sum(p.numel() for p in existing_model.parameters() if p.requires_grad):,}")
    print(f"   Model info: {model_info}")
    
print(f"🔄 Needs retraining: {needs_retraining}")

## 4. Load Dataset with Fixed Class Mappings

In [ ]:
# Create datasets with CORRECTED class mappings
train_transforms, val_test_transforms = get_transforms()

print("📂 Loading datasets with corrected class mappings...")

try:
    # Load with our custom class mapping
    train_dataset = CustomImageFolder(
        TRAIN_PATH, 
        transform=train_transforms,
        class_to_idx=DATASET_CONFIG['class_to_idx']
    )
    
    val_dataset = CustomImageFolder(
        VAL_PATH, 
        transform=val_test_transforms,
        class_to_idx=DATASET_CONFIG['class_to_idx']
    )
    
    test_dataset = CustomImageFolder(
        TEST_PATH, 
        transform=val_test_transforms,
        class_to_idx=DATASET_CONFIG['class_to_idx']
    )
    
    print(f"✅ Datasets loaded successfully:")
    print(f"   Training: {len(train_dataset):,} samples")
    print(f"   Validation: {len(val_dataset):,} samples") 
    print(f"   Test: {len(test_dataset):,} samples")
    
    # Verify class mappings are correct
    print(f"\n🔍 Verifying class mappings:")
    print(f"   Train classes: {train_dataset.classes[:5]}...")
    print(f"   Class to idx sample: {dict(list(train_dataset.class_to_idx.items())[:3])}")
    
    # Check mango class distribution
    train_targets = np.array(train_dataset.targets)
    mango_indices = [DATASET_CONFIG['class_to_idx'][cls] for cls in DATASET_CONFIG['class_to_idx'] if 'mango' in cls]
    mango_samples = sum([np.sum(train_targets == idx) for idx in mango_indices])
    print(f"   🥭 Mango samples in training: {mango_samples}")
    
    # Create data loaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=True, 
        num_workers=CONFIG['num_workers'],
        pin_memory=True
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=False, 
        num_workers=CONFIG['num_workers'],
        pin_memory=True
    )
    
    test_loader = DataLoader(
        test_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=False, 
        num_workers=CONFIG['num_workers'],
        pin_memory=True
    )
    
    print(f"✅ Data loaders created: {len(train_loader)} train batches, {len(val_loader)} val batches")
    
    datasets_loaded = True
    
except Exception as e:
    print(f"❌ Error loading datasets: {e}")
    print("📝 This might be because we're not in Kaggle environment.")
    print("   In Kaggle, make sure to:")
    print("   1. Upload your dataset as a Kaggle dataset")
    print("   2. Add it as input to your notebook")
    print("   3. Update the DATASET_PATH variable above")
    
    datasets_loaded = False

## 5. Evaluate Existing Model Performance

In [ ]:
def evaluate_model(model, dataloader, device, class_names, split_name=""):
    """Comprehensive model evaluation"""
    model.eval()
    all_preds = []
    all_targets = []
    all_probs = []
    
    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(dataloader):
            data, target = data.to(device), target.to(device)
            output = model(data)
            probs = torch.softmax(output, dim=1)
            _, predicted = torch.max(output, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(target.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            
            if batch_idx % 20 == 0:
                print(f"   Evaluating batch {batch_idx}/{len(dataloader)}", end='\r')
    
    # Calculate metrics
    accuracy = accuracy_score(all_targets, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_targets, all_preds, average='weighted')
    
    print(f"\n📊 {split_name} Results:")
    print(f"   Accuracy: {accuracy:.4f}")
    print(f"   Precision: {precision:.4f}")
    print(f"   Recall: {recall:.4f}")  
    print(f"   F1-Score: {f1:.4f}")
    
    # Detailed classification report
    print(f"\n📋 Detailed Classification Report ({split_name}):")
    report = classification_report(all_targets, all_preds, target_names=class_names, output_dict=True)
    
    # Focus on mango classes
    mango_metrics = {}
    print(f"\n🥭 Mango Classification Performance:")
    for class_name in class_names:
        if 'mango' in class_name:
            if class_name in report:
                metrics = report[class_name]
                mango_metrics[class_name] = metrics
                print(f"   {class_name:<20}: Precision={metrics['precision']:.3f}, Recall={metrics['recall']:.3f}, F1={metrics['f1-score']:.3f}")
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'predictions': all_preds,
        'targets': all_targets,
        'probabilities': all_probs,
        'mango_metrics': mango_metrics,
        'full_report': report
    }

def plot_confusion_matrix(targets, predictions, class_names, title="Confusion Matrix"):
    """Plot confusion matrix with special focus on mango classes"""
    cm = confusion_matrix(targets, predictions)
    
    plt.figure(figsize=(14, 12))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Count'})
    plt.title(f'{title}\nOverall Accuracy: {accuracy_score(targets, predictions):.3f}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    
    # Highlight mango confusion
    mango_indices = [i for i, name in enumerate(class_names) if 'mango' in name]
    for i in mango_indices:
        for j in mango_indices:
            if i != j and cm[i, j] > 0:
                plt.text(j + 0.5, i + 0.7, '❌', ha='center', va='center', 
                        color='red', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Mango-specific confusion matrix
    if len(mango_indices) > 1:
        mango_cm = cm[np.ix_(mango_indices, mango_indices)]
        mango_names = [class_names[i] for i in mango_indices]
        
        plt.figure(figsize=(8, 6))
        sns.heatmap(mango_cm, annot=True, fmt='d', cmap='Reds',
                    xticklabels=mango_names, yticklabels=mango_names)
        plt.title('🥭 Mango Classes Confusion Matrix')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.xticks(rotation=45, ha='right')
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.show()

# Evaluate existing model if we have both model and data
if existing_model and datasets_loaded:
    device = torch.device(CONFIG['device'])
    existing_model = existing_model.to(device)
    
    print("🔍 Evaluating existing model performance...")
    print("=" * 60)
    
    # Evaluate on validation set
    val_results = evaluate_model(
        existing_model, val_loader, device, 
        list(DATASET_CONFIG['classes'].values()), 
        "Validation"
    )
    
    # Plot confusion matrix
    plot_confusion_matrix(
        val_results['targets'], 
        val_results['predictions'],
        list(DATASET_CONFIG['classes'].values()),
        "Existing Model - Validation Set"
    )
    
    # Determine if retraining is needed based on performance
    PERFORMANCE_THRESHOLD = 0.85  # 85% accuracy threshold
    MANGO_F1_THRESHOLD = 0.80     # 80% F1 for mango classes
    
    mango_f1_avg = np.mean([metrics['f1-score'] for metrics in val_results['mango_metrics'].values()])
    
    print(f"\n🎯 Performance Analysis:")
    print(f"   Overall accuracy: {val_results['accuracy']:.3f} (threshold: {PERFORMANCE_THRESHOLD})")
    print(f"   Mango F1 average: {mango_f1_avg:.3f} (threshold: {MANGO_F1_THRESHOLD})")
    
    performance_based_retraining = (
        val_results['accuracy'] < PERFORMANCE_THRESHOLD or 
        mango_f1_avg < MANGO_F1_THRESHOLD
    )
    
    final_needs_retraining = needs_retraining or performance_based_retraining
    
    print(f"\n🔄 Retraining Decision:")
    print(f"   Config issues: {needs_retraining}")
    print(f"   Performance issues: {performance_based_retraining}")
    print(f"   FINAL DECISION: {'RETRAIN' if final_needs_retraining else 'USE EXISTING'}")
    
else:
    print("⏭️  Skipping evaluation - will proceed to training")
    final_needs_retraining = True
    val_results = None

## 6. Model Retraining (Conditional)

In [ ]:
# CONDITIONAL RETRAINING - Only if needed
if final_needs_retraining and datasets_loaded:
    print("🚀 STARTING MODEL RETRAINING")
    print("=" * 60)
    print(f"Reason: {'Config/mapping issues' if needs_retraining else 'Performance below threshold'}")
    
    # Create fresh model
    model = create_model(DATASET_CONFIG['num_classes'], CONFIG['model_type'])
    device = torch.device(CONFIG['device'])
    model = model.to(device)
    
    # Loss function with class weights to handle imbalance
    class_counts = np.bincount([train_dataset.targets[i] for i in range(len(train_dataset.targets))])
    class_weights = 1.0 / (class_counts + 1e-6)
    class_weights = class_weights / class_weights.sum() * len(class_weights)
    
    print(f"📊 Class weights (to handle imbalance):")
    for i, (class_name, weight) in enumerate(zip(DATASET_CONFIG['classes'].values(), class_weights)):
        if 'mango' in class_name:  # Show mango weights specifically
            print(f"   {class_name}: {weight:.3f}")
    
    criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor(class_weights).to(device))
    
    # Optimizer with different learning rates for different parts
    backbone_params = []
    classifier_params = []
    
    for name, param in model.named_parameters():
        if 'fc' in name or 'classifier' in name:
            classifier_params.append(param)
        else:
            backbone_params.append(param)
    
    optimizer = optim.AdamW([
        {'params': backbone_params, 'lr': CONFIG['learning_rate'] * 0.1},  # Lower LR for pretrained layers
        {'params': classifier_params, 'lr': CONFIG['learning_rate']}       # Higher LR for new classifier
    ], weight_decay=1e-4)
    
    # Learning rate scheduler
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, 
        max_lr=CONFIG['learning_rate'],
        epochs=CONFIG['epochs'],
        steps_per_epoch=len(train_loader),
        pct_start=0.3,
        anneal_strategy='cos'
    )
    
    print(f"✅ Training setup complete:")
    print(f"   Model: {CONFIG['model_type']} with {sum(p.numel() for p in model.parameters()):,} parameters")
    print(f"   Optimizer: AdamW with differential learning rates")
    print(f"   Scheduler: OneCycleLR")
    print(f"   Loss: CrossEntropyLoss with class weights")
    
else:
    print("✅ SKIPPING RETRAINING")
    if not final_needs_retraining:
        print("   Reason: Existing model performance is satisfactory")
    else:
        print("   Reason: Dataset not loaded (likely not in Kaggle environment)")
    
    model = existing_model

In [ ]:
# Training loop (only if retraining)
if final_needs_retraining and datasets_loaded:
    
    def train_epoch(model, dataloader, criterion, optimizer, scheduler, device, epoch):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        progress_bar = range(len(dataloader))
        
        for batch_idx, (data, target) in enumerate(dataloader):
            data, target = data.to(device), target.to(device)
            
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            
            # Gradient clipping for stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            scheduler.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
            
            if batch_idx % 20 == 0:
                current_lr = scheduler.get_last_lr()[0]
                print(f'   Epoch {epoch+1}, Batch {batch_idx}/{len(dataloader)}, '
                      f'Loss: {loss.item():.4f}, LR: {current_lr:.6f}', end='\r')
        
        epoch_loss = running_loss / len(dataloader)
        epoch_acc = 100. * correct / total
        return epoch_loss, epoch_acc
    
    def validate_epoch(model, dataloader, criterion, device):
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for data, target in dataloader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                val_loss += criterion(output, target).item()
                
                _, predicted = torch.max(output.data, 1)
                total += target.size(0)
                correct += (predicted == target).sum().item()
        
        val_loss /= len(dataloader)
        val_acc = 100. * correct / total
        return val_loss, val_acc
    
    # Training history tracking
    train_losses, train_accs = [], []
    val_losses, val_accs = [], []
    best_val_acc = 0.0
    best_model_state = None
    
    print(f"\n🚀 Starting training for {CONFIG['epochs']} epochs...")
    start_time = time.time()
    
    for epoch in range(CONFIG['epochs']):
        print(f'\n📊 Epoch {epoch+1}/{CONFIG["epochs"]}')
        print('-' * 50)
        
        # Train
        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, scheduler, device, epoch
        )
        
        # Validate
        val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)
        
        # Save history
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        
        print(f'\n   Train: Loss={train_loss:.4f}, Acc={train_acc:.2f}%')
        print(f'   Val:   Loss={val_loss:.4f}, Acc={val_acc:.2f}%')
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
            print(f'   🎯 New best validation accuracy: {val_acc:.2f}%')
            
        # Early stopping check
        if epoch > 10 and val_acc < best_val_acc - 5:  # Stop if val acc drops by 5%
            print(f"   ⏹️  Early stopping triggered")
            break
    
    training_time = time.time() - start_time
    print(f"\n✅ Training completed in {training_time/3600:.2f} hours")
    print(f"🏆 Best validation accuracy: {best_val_acc:.2f}%")
    
    # Load best model
    if best_model_state:
        model.load_state_dict(best_model_state)
        print("✅ Best model loaded")
    
    retrained_model = model
    training_history = {
        'train_losses': train_losses,
        'train_accs': train_accs,
        'val_losses': val_losses,
        'val_accs': val_accs,
        'best_val_acc': best_val_acc,
        'training_time': training_time
    }
    
else:
    print("⏭️  Training skipped")
    retrained_model = model if 'model' in locals() else existing_model
    training_history = None

## 7. Model Comparison and Validation

In [ ]:
# Comprehensive evaluation of final model
if datasets_loaded:
    print("🔍 FINAL MODEL EVALUATION")
    print("=" * 60)
    
    device = torch.device(CONFIG['device'])
    final_model = retrained_model.to(device)
    
    # Evaluate on test set
    test_results = evaluate_model(
        final_model, test_loader, device,
        list(DATASET_CONFIG['classes'].values()),
        "Test Set"
    )
    
    # Plot final confusion matrix
    plot_confusion_matrix(
        test_results['targets'],
        test_results['predictions'], 
        list(DATASET_CONFIG['classes'].values()),
        "Final Model - Test Set Performance"
    )
    
    # Training history plots (if we retrained)
    if training_history and final_needs_retraining:
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        
        # Loss plot
        axes[0].plot(training_history['train_losses'], label='Training Loss', color='blue', alpha=0.7)
        axes[0].plot(training_history['val_losses'], label='Validation Loss', color='red', alpha=0.7)
        axes[0].set_title('Training and Validation Loss')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Accuracy plot
        axes[1].plot(training_history['train_accs'], label='Training Accuracy', color='blue', alpha=0.7)
        axes[1].plot(training_history['val_accs'], label='Validation Accuracy', color='red', alpha=0.7)
        axes[1].set_title('Training and Validation Accuracy')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Accuracy (%)')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    # Detailed mango analysis
    print(f"\n🥭 DETAILED MANGO ANALYSIS:")
    print("=" * 40)
    
    mango_classes = ['mango_unripe', 'mango_early_ripe', 'mango_partially_ripe', 'mango_ripe', 'mango_rotten']
    mango_indices = [DATASET_CONFIG['class_to_idx'][cls] for cls in mango_classes]
    
    for i, class_name in enumerate(mango_classes):
        class_idx = DATASET_CONFIG['class_to_idx'][class_name]
        
        # Get predictions for this class
        class_mask = np.array(test_results['targets']) == class_idx
        if np.sum(class_mask) > 0:
            class_preds = np.array(test_results['predictions'])[class_mask]
            class_targets = np.array(test_results['targets'])[class_mask]
            
            accuracy = accuracy_score(class_targets, class_preds)
            
            # Most common wrong predictions
            wrong_preds = class_preds[class_preds != class_idx]
            if len(wrong_preds) > 0:
                wrong_classes, counts = np.unique(wrong_preds, return_counts=True)
                most_confused_idx = wrong_classes[np.argmax(counts)]
                most_confused_class = DATASET_CONFIG['classes'][most_confused_idx]
                confusion_rate = counts[np.argmax(counts)] / len(class_preds)
            else:
                most_confused_class = "None"
                confusion_rate = 0.0
            
            print(f"   {class_name:<20}: Accuracy={accuracy:.3f}, Most confused with: {most_confused_class} ({confusion_rate:.1%})")
    
    # Compare with original results if available
    if val_results is not None:
        print(f"\n📈 PERFORMANCE COMPARISON:")
        print("=" * 40)
        print(f"   Original validation accuracy: {val_results['accuracy']:.3f}")
        print(f"   New test accuracy:           {test_results['accuracy']:.3f}")
        print(f"   Improvement:                 {test_results['accuracy'] - val_results['accuracy']:+.3f}")
        
        # Mango-specific comparison
        orig_mango_f1 = np.mean([metrics['f1-score'] for metrics in val_results['mango_metrics'].values()])
        new_mango_f1 = np.mean([metrics['f1-score'] for metrics in test_results['mango_metrics'].values()])
        print(f"   Original mango F1:           {orig_mango_f1:.3f}")  
        print(f"   New mango F1:                {new_mango_f1:.3f}")
        print(f"   Mango improvement:           {new_mango_f1 - orig_mango_f1:+.3f}")
        
        if test_results['accuracy'] > val_results['accuracy']:
            print("   🎉 NEW MODEL IS BETTER!")
        else:
            print("   ⚠️  New model performance is similar/worse")
    
    final_results = test_results
    
else:
    print("⏭️  Final evaluation skipped - no datasets loaded")
    final_results = None

## 8. Save Improved Model

In [ ]:
# Save the final model with metadata
if 'final_model' in locals():
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Prepare model metadata
    model_metadata = {
        'model_state_dict': final_model.state_dict() if hasattr(final_model, 'state_dict') else final_model,
        'model_type': CONFIG['model_type'],
        'num_classes': DATASET_CONFIG['num_classes'],
        'class_names': list(DATASET_CONFIG['classes'].values()),
        'class_to_idx': DATASET_CONFIG['class_to_idx'],
        'training_config': CONFIG,
        'timestamp': timestamp,
        'pytorch_version': torch.__version__,
    }
    
    # Add performance metrics if available
    if final_results:
        model_metadata.update({
            'test_accuracy': final_results['accuracy'],
            'test_precision': final_results['precision'],
            'test_recall': final_results['recall'],
            'test_f1': final_results['f1'],
            'mango_metrics': final_results['mango_metrics']
        })
        
    # Add training history if available
    if training_history:
        model_metadata.update({
            'training_history': training_history,
            'epochs_trained': len(training_history['train_losses']),
            'best_val_acc': training_history['best_val_acc'],
            'training_time_hours': training_history['training_time'] / 3600
        })
    
    # Save model files
    model_filename = f"fruit_classification_fixed_{timestamp}.pth"
    
    try:
        torch.save(model_metadata, model_filename)
        print(f"✅ Model saved successfully: {model_filename}")
        print(f"📁 File size: {os.path.getsize(model_filename) / (1024*1024):.1f} MB")
        
        # Also save in the standard name for easy loading
        standard_filename = "classification_best_fixed.pth"
        torch.save(model_metadata, standard_filename)
        print(f"✅ Model also saved as: {standard_filename}")
        
        # Save just the state dict for deployment
        deployment_filename = f"classification_deployment_{timestamp}.pth"
        torch.save(final_model.state_dict(), deployment_filename)
        print(f"✅ Deployment model saved: {deployment_filename}")
        
    except Exception as e:
        print(f"❌ Error saving model: {e}")
        
    # Create a summary report
    report_filename = f"training_report_{timestamp}.txt"
    
    with open(report_filename, 'w') as f:
        f.write("FRESH ML - Fruit Classification Training Report\\n")
        f.write("=" * 60 + "\\n\\n")
        
        f.write("CRITICAL ISSUES FIXED:\\n")
        f.write("- ❌ Class mapping mismatch (ImageFolder vs dataset.yaml order)\\n")
        f.write("- ❌ Config key errors (nc vs num_classes, names vs classes)\\n")
        f.write("- ❌ Green mango misclassification due to wrong indices\\n\\n")
        
        f.write("DATASET CONFIGURATION:\\n")
        f.write(f"- Classes: {DATASET_CONFIG['num_classes']}\\n")
        f.write(f"- Model: {CONFIG['model_type']}\\n")
        f.write(f"- Training samples: {len(train_dataset) if 'train_dataset' in locals() else 'N/A'}\\n\\n")
        
        if final_results:
            f.write(f"FINAL PERFORMANCE:\\n")
            f.write(f"- Test Accuracy: {final_results['accuracy']:.3f}\\n")
            f.write(f"- Test F1-Score: {final_results['f1']:.3f}\\n")
            f.write(f"- Mango Classes F1: {np.mean([m['f1-score'] for m in final_results['mango_metrics'].values()]):.3f}\\n\\n")
            
        if training_history:
            f.write(f"TRAINING DETAILS:\\n")
            f.write(f"- Epochs: {len(training_history['train_losses'])}\\n")
            f.write(f"- Best Val Acc: {training_history['best_val_acc']:.3f}\\n")
            f.write(f"- Training Time: {training_history['training_time']/3600:.2f} hours\\n\\n")
            
        f.write(f"FILES CREATED:\\n")
        f.write(f"- Full model: {model_filename}\\n")
        f.write(f"- Standard model: {standard_filename}\\n")
        f.write(f"- Deployment model: {deployment_filename}\\n")
        f.write(f"- This report: {report_filename}\\n")
        
    print(f"✅ Training report saved: {report_filename}")
    
    # Final summary
    print(f"\\n🎉 TRAINING COMPLETE!")
    print("=" * 60)
    print(f"✅ Issues Fixed: Class mapping, config keys, mango classification")
    print(f"✅ Model Type: {CONFIG['model_type']} with {DATASET_CONFIG['num_classes']} classes")
    if final_results:
        print(f"✅ Test Accuracy: {final_results['accuracy']:.1%}")
        mango_f1 = np.mean([m['f1-score'] for m in final_results['mango_metrics'].values()])
        print(f"✅ Mango F1-Score: {mango_f1:.1%}")
    
    print(f"\\n🚀 Ready for deployment!")
    print(f"   Use '{standard_filename}' in your production API")
    print(f"   The class mappings are now correctly aligned with your dataset.yaml")
    
else:
    print("⏭️  Model saving skipped - no final model available")
    print("   This is normal if running outside Kaggle environment")

## 🎯 Summary and Next Steps

### Critical Issues Identified and Fixed:

1. **🚨 Class Mapping Mismatch** - The biggest issue!
   - **Problem**: Model trained with ImageFolder alphabetical order (`grapefruit_pink=0, mango_unripe=11`)
   - **Expected**: Custom dataset.yaml order (`mango_unripe=0, mango_ripe=3`)
   - **Impact**: Every prediction was mapped to wrong class
   - **Solution**: CustomImageFolder class with explicit class_to_idx mapping

2. **🔧 Configuration Key Errors**
   - **Problem**: Training script used `config['nc']` and `config['names']`
   - **Reality**: dataset.yaml has `config['num_classes']` and `config['classes']`
   - **Solution**: Fixed keys in this notebook

3. **🥭 Mango Classification Issues**
   - **Root Cause**: Class mapping mismatch made all mango predictions wrong
   - **Green mangoes → "ripe"**: Because model's class 11 (mango_unripe) was interpreted as class 3 (mango_ripe)
   - **Solution**: Proper class alignment fixes this automatically

### 🚀 Instructions for Kaggle:

1. **Upload your dataset** to Kaggle as a dataset
2. **Update paths** in cell 4: Change `DATASET_PATH` to your Kaggle dataset path
3. **Add GPU**: Runtime → Change runtime type → GPU (T4 or P100)
4. **Run all cells**: The notebook will automatically detect and fix issues
5. **Download models**: After training, download the `.pth` files for deployment

### 🎯 Expected Results:

- **Fixes green mango misclassification** by using correct class indices
- **Improves overall accuracy** with proper data augmentation and class weighting
- **Provides detailed mango-specific metrics** to track ripeness classification
- **Saves properly formatted model** compatible with your existing API

### 📁 Files Generated:

- `classification_best_fixed.pth` - Main model for deployment
- `classification_deployment_*.pth` - State dict only for production
- `training_report_*.txt` - Detailed training summary

**This notebook will solve your green mango classification problem!** 🥭✅